# Build combined_legal_cases.json on Google Colab

1. **Upload PDFs**: Upload a ZIP of your NLR/SLR PDFs, or mount Google Drive.
2. Run the cells below.
3. Download `combined_legal_cases.json` and put it in your project: `data/processed/`.

## 1. Install pdfplumber

In [ ]:
!pip install -q pdfplumber

## 2. Get your PDFs into Colab

**Option A – Upload a ZIP:**  
Run the next cell, then click "Choose Files" and select your ZIP. It will unzip to `/content/pdfs`.

**Option B – Google Drive:**  
If your PDFs are in Drive, run the "Mount Drive" cell instead and set `PDF_DIR` in the config cell to your folder (e.g. `/content/drive/MyDrive/raw_documents/NLR_All_Volumes all`).

In [ ]:
# Option A: Upload ZIP and unzip
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # choose your ZIP
zip_path = list(uploaded.keys())[0]
Path("/content/pdfs").mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/pdfs")
print("Done. PDFs under /content/pdfs")

In [ ]:
# Option B: Mount Google Drive (if PDFs are in Drive)
from google.colab import drive
drive.mount("/content/drive")
# Then set PDF_DIR in the next cell to e.g. 
# Path("/content/drive/MyDrive/raw_documents/NLR_All_Volumes all")

## 3. Config & build corpus

In [ ]:
import json
import re
import time
from pathlib import Path

# ----- EDIT IF NEEDED -----
PDF_DIR = Path("/content/pdfs")   # or Path("/content/drive/MyDrive/...")
MAX_FILES = 2000
MAX_CHARS = 5000
OUT_FILE = Path("/content/combined_legal_cases.json")
# --------------------------

def collect_pdfs(root, max_files):
    out = []
    if not root.exists():
        return out
    for path in root.rglob("*.pdf"):
        out.append(path)
        if len(out) >= max_files:
            return out
    return out

def extract_one(pdf_path, max_chars=5000):
    import pdfplumber
    text_parts, total = [], 0
    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages[:30]:
                if total >= max_chars:
                    break
                t = page.extract_text()
                if t:
                    text_parts.append(t)
                    total += len(t)
    except Exception:
        return None
    raw = "\n".join(text_parts)
    if len(raw.strip()) < 100:
        return None
    raw = raw[:max_chars]
    cleaned = re.sub(r"\s+", " ", raw).strip()
    return {"file_name": pdf_path.name, "raw_text": raw, "cleaned_text": cleaned}

paths = collect_pdfs(PDF_DIR, MAX_FILES)
print(f"Found {len(paths)} PDFs")
if not paths:
    raise SystemExit("No PDFs. Check PDF_DIR or upload ZIP.")

t0 = time.perf_counter()
cases = []
for i, p in enumerate(paths):
    if (i + 1) % 200 == 0 or i == 0:
        print(f"  {i + 1}/{len(paths)}")
    rec = extract_one(p, MAX_CHARS)
    if rec:
        cases.append(rec)
elapsed = time.perf_counter() - t0
print(f"Extracted {len(cases)} cases in {elapsed:.1f}s")

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump({"cases": cases, "built_from_raw_documents": True}, f, ensure_ascii=False, indent=0)
print(f"Saved: {OUT_FILE}")

## 4. Download the JSON

In [ ]:
from google.colab import files
files.download("/content/combined_legal_cases.json")
print("Download started. Put the file in your project: data/processed/combined_legal_cases.json")